In [7]:
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv

repo = Path(find_dotenv(usecwd=True)).parent
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(repo / "data" / "processed" / "orders_enriched.csv", parse_dates=date_cols)

print("shape:", orders.shape)

shape: (99441, 18)


In [8]:
# one row per person: their first order, delivered, with delivery info
m = orders[(orders["order_number"] == 1)
           & (orders["order_status"] == "delivered")
           & orders["is_late"].notna()
           & (orders["order_purchase_timestamp"] >= "2017-01-01")
           & (orders["order_purchase_timestamp"] < "2018-03-01")].copy()   # 6+ months to return

m["repeat"] = (m["person_total_orders"] > 1).astype(int)         # target: ordered again

features = ["order_value", "n_items", "delivery_days", "delay_days",
            "is_late", "review_score", "customer_state"]

print("people:", len(m), " repeat buyers:", m["repeat"].sum(), " rate %:", round(m["repeat"].mean() * 100, 2))
print(m[features].isna().sum())

people: 55227  repeat buyers: 2220  rate %: 4.02
order_value         0
n_items             0
delivery_days       0
delay_days          0
is_late             0
review_score      409
customer_state      0
dtype: int64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

X = m[features].copy()
X["is_late"] = X["is_late"].astype(int)
y = m["repeat"]

# 80/20 split; stratify keeps the 4% repeat share equal in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

num_cols = ["order_value", "n_items", "delivery_days", "delay_days", "is_late", "review_score"]
cat_cols = ["customer_state"]

prep = ColumnTransformer([
    ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

logit = Pipeline([("prep", prep),
                  ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
logit.fit(X_train, y_train)

proba = logit.predict_proba(X_test)[:, 1]
print("train rows:", len(X_train), " test rows:", len(X_test))
print("repeat share train / test:", round(y_train.mean() * 100, 2), "/", round(y_test.mean() * 100, 2))
print("always-predict-no accuracy:", round((1 - y_test.mean()) * 100, 2))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, logit.predict(X_test), digits=3))

train rows: 44181  test rows: 11046
repeat share train / test: 4.02 / 4.02
always-predict-no accuracy: 95.98
ROC-AUC: 0.539
              precision    recall  f1-score   support

           0      0.963     0.507     0.665     10602
           1      0.044     0.541     0.081       444

    accuracy                          0.509     11046
   macro avg      0.504     0.524     0.373     11046
weighted avg      0.927     0.509     0.641     11046



**Insight (baseline logistic regression):** On 55,227 first orders with at least 6 months to return (4.02% repeat buyers; 80/20 stratified split, 11,046 test rows), the model reaches ROC-AUC 0.539, barely above the 0.5 of random guessing. For repeat buyers, precision is 0.044 against a base rate of 4.02%, and recall is 0.541, reached by flagging about half of all customers. Accuracy is 50.9%, but a model that always predicts "no repeat" scores 95.98%, so accuracy is not a useful metric here.

**Caution:** Only 444 repeat buyers are in the test set, so the metrics are noisy. The features (order value, items, delivery, review score, state) describe the first order only, and I have no marketing, price or customer-history data.

**Business implication (hypothesis):** Repeat buying is hard to predict from first-order data. This fits the earlier tests, where delivery and review score barely moved the repeat rate.

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([("prep", prep),
               ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=50,
                                                class_weight="balanced_subsample",
                                                random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)

proba_rf = rf.predict_proba(X_test)[:, 1]   # probability of class 1 (repeat buyer)
print("Random forest ROC-AUC:", round(roc_auc_score(y_test, proba_rf), 3))
print("Logistic regression ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

Random forest ROC-AUC: 0.533
Logistic regression ROC-AUC: 0.539


**Insight (model comparison):** Logistic regression (ROC-AUC 0.539) and a random forest (0.533) both rank repeat buyers only slightly better than random guessing (0.5). The 0.006 gap is smaller than the noise expected from 444 repeat buyers in the test set, so neither model is better. The features describe the first order only (value, items, delivery, review score, state).

**Business implication (hypothesis):** Repeat buying cannot be predicted from first-order data, so a scoring model built on these features would not help target customers. This fits the earlier tests, where delivery and review score had small or no links to repeat buying. Better signals would need data I do not have, such as marketing contact, prices, promotions or browsing behaviour.

In [11]:
from sklearn.inspection import permutation_importance

result = permutation_importance(rf, X_test, y_test, scoring="roc_auc",
                                n_repeats=10, random_state=42, n_jobs=-1)

imp = pd.DataFrame({"feature": X_test.columns,
                    "auc_drop": result.importances_mean,
                    "std": result.importances_std}).sort_values("auc_drop", ascending=False)
print(imp.round(4).to_string(index=False))

       feature  auc_drop    std
    delay_days    0.0059 0.0054
  review_score    0.0044 0.0052
       n_items    0.0042 0.0077
   order_value    0.0040 0.0080
       is_late    0.0017 0.0017
 delivery_days    0.0012 0.0044
customer_state   -0.0074 0.0137


**Insight (permutation importance, random forest, ROC-AUC):** Shuffling any single feature changes the test AUC by at most 0.0059 (delay_days), and every change is within about one standard deviation of zero (for example review_score 0.0044 with std 0.0052). customer_state is negative (-0.0074), so it adds noise rather than signal. No feature clearly drives the prediction, so I do not rank them. This fits the weak model (AUC 0.533) and the earlier tests, where delivery and review score had small or no links to repeat buying.

## Limitations of the model
- **Weak signal:** ROC-AUC is 0.539 (logistic regression) and 0.533 (random forest), close to random guessing (0.5). First-order data barely predicts who returns.
- **Class imbalance:** only 4.02% of the 55,227 people are repeat buyers (444 in the test set), so the metrics are noisy. I used one 80/20 split and did not run cross-validation or compute confidence intervals.
- **Accuracy is misleading:** always predicting "no repeat" scores 95.98%, so I judged the models on ROC-AUC, precision and recall for the repeat group.
- **Features:** only first-order information (value, items, delivery, review score, state). I have no marketing, price, promotion or browsing data, and I did not use product category.
- **Fair target:** only first orders with at least 6 months to return (2017-01 to 2018-02) are included, so the model describes that group only.
- **Not tuned:** I did not tune the models, because a larger effort would not fix the missing signal.
- **Association, not cause:** nothing here shows that changing delivery or reviews would change repeat buying.

**Takeaway:** repeat buying cannot be predicted from first-order data, so a scoring model built on these features would not help target customers. Better retention work needs different data, such as marketing contact and price information, or experiments like the win-back test on the "At risk" segment.

In [12]:
from scipy import stats
import numpy as np

# Query 19 output: category, customers, repeat rate %
q19 = [("fashion_bags_accessories",1726,5.91),("furniture_decor",6039,4.55),("bed_bath_table",8847,4.51),
       ("sports_leisure",7326,3.92),("garden_tools",3383,2.99),("pet_shop",1646,2.98),
       ("computers_accessories",6404,2.97),("perfumery",3047,2.86),("(no category)",1377,2.83),
       ("health_beauty",8495,2.79),("housewares",5604,2.73),("telephony",4079,2.45),
       ("toys",3723,2.34),("baby",2747,2.15),("watches_gifts",5422,2.14),("stationery",2231,2.11),
       ("auto",3762,2.05),("office_furniture",1237,2.02),("cool_stuff",3529,1.87),
       ("consoles_games",1031,1.75),("electronics",2489,1.65)]

cat = pd.DataFrame(q19, columns=["category", "customers", "rate_pct"])
cat["repeat"] = (cat["customers"] * cat["rate_pct"] / 100).round().astype(int)   # rebuild counts
cat["no_repeat"] = cat["customers"] - cat["repeat"]

chi2_c, p_c, dof_c, _ = stats.chi2_contingency(cat[["repeat", "no_repeat"]].values)
n = cat["customers"].sum()
print("customers:", n, " repeat buyers:", cat["repeat"].sum(), " rate %:", round(cat["repeat"].sum() / n * 100, 2))
print("chi2:", round(chi2_c, 1), " dof:", dof_c, " p-value:", p_c)
print("Cramer's V:", round((chi2_c / n) ** 0.5, 3))

customers: 84144  repeat buyers: 2555  rate %: 3.04
chi2: 279.4  dof: 20  p-value: 1.2933223662822915e-47
Cramer's V: 0.058


**Insight (category vs repeat buying):** Across 21 first-item categories (84,144 customers, 2,555 repeat buyers, 3.04%), the repeat rate differs far more than chance allows (chi-square 279.4, dof 20, p < 0.001). The effect is small (Cramer's V = 0.058). Fashion_bags_accessories is highest at 5.91% (102 of 1,726), about double the rest (2.98%); furniture_decor (4.55%) and bed_bath_table (4.51%) also return above average, and electronics (1.65%) is lowest. The remaining categories still differ without fashion bags, so it is not one outlier.

**Caution:** Association only. Categories differ in price, buyer type and purchase purpose (a mattress is a one-off, a bag may be a gift), and I did not control for any of it. The counts were rebuilt from Query 19's rates. The categories are the 21 rows with 1,000+ customers plus a blank-category group of 1,377 customers, so not every customer is included.

## Answer to the business question: why don't customers come back?

**What the data shows**
1. **Not returning is the default, at every age.** 96.96% of 94,990 customers bought once, and in no cohort does any month after the first exceed 0.71%.
2. **Delivery and satisfaction explain little.** Late first orders return slightly less (3.07% vs 2.53%, p = 0.0088; 4.09% vs 3.22% with 6+ months to return, p = 0.0063). Review score shows no measurable link (2.69% to 3.15%, p = 0.082).
3. **What customers buy first matters more.** The repeat rate ranges from 1.65% (electronics) to 5.91% (fashion_bags_accessories), p < 0.001, but the effect is small (Cramer's V = 0.058).
4. **First-order data cannot predict who returns.** Logistic regression and a random forest reach ROC-AUC 0.539 and 0.533, close to guessing.
5. **The real reasons are not visible in this dataset.** There is no marketing, price, promotion or competitor data, so I can describe who returns but not why.

**What the business could test (hypotheses, not conclusions)**
- A win-back experiment on the "At risk" segment (14.89% of customers, 26.52% of spend), with a control group.
- Logistics improvements in MA, CE, BA and RJ, to protect review scores.
- First-purchase offers and cross-selling that start from the higher-repeat categories (fashion_bags_accessories, furniture_decor, bed_bath_table).
- Do not expect on-time delivery alone to fix retention.

**Size of the prize (illustration, not a forecast):** a 1-point rise in the repeat rate would mean about 950 more repeat buyers out of 94,990 (0.01 x 94,990). At the average order value of 159.83, one extra order each would be about 151,800 in revenue. That assumes the rise is achievable and that each repeat buyer places one more average order, and I have no evidence for either.

In [13]:
out = repo / "data" / "processed"

cat_out = cat[["category", "customers", "repeat", "no_repeat"]].copy()
cat_out["repeat_rate_pct"] = (cat_out["repeat"] / cat_out["customers"] * 100).round(2)
cat_out.to_csv(out / "category_repeat.csv", index=False)

print("category_repeat.csv rows:", len(cat_out))
print(cat_out.head(3).to_string(index=False))

category_repeat.csv rows: 21
                category  customers  repeat  no_repeat  repeat_rate_pct
fashion_bags_accessories       1726     102       1624             5.91
         furniture_decor       6039     275       5764             4.55
          bed_bath_table       8847     399       8448             4.51
